# Scanorama best - cellranger_versions

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd
import scanorama
import scanpy as sc
import scipy.sparse as sp

from scib.metrics.metrics import metrics


In [2]:
DATASET_NAME = 'cellranger_versions'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'batch'
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
N_TOP_HVG = 2000
INTERNAL_DIMRED = 50
BATCH_SIZE = 5000
APPROX = True
UNION = False

EMBED_KEY = "X_scanorama"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scanorama_metric_batch"
METRIC_LABEL_KEY = "scanorama_metric_label"

assert DATASET_NAME == 'cellranger_versions'
assert BATCH_KEY == 'batch'
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "internal_dimred": INTERNAL_DIMRED,
        "batch_size": BATCH_SIZE,
        "n_top_hvg": N_TOP_HVG,
    },
    name="configuration",
)


dataset                                          cellranger_versions
data_path          /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                      batch
label_key                                                  cell_type
metrics                     [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                                 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
internal_dimred                                                   50
batch_size                                                      5000
n_top_hvg                                                       2000
Name: configuration, dtype: object

In [3]:
def validate_dataset_contents():
    adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
    try:
        required_keys = (BATCH_KEY, LABEL_KEY)
        missing_keys = [
            key for key in required_keys if key not in adata_backed.obs
        ]
        if missing_keys:
            raise ValueError(
                f"Missing obs keys {missing_keys}. "
                f"Available keys: {list(adata_backed.obs.columns)}"
            )

        obs = adata_backed.obs[[BATCH_KEY, LABEL_KEY]].copy()
        pseudotime = None
        if "TRAJ" in SELECTED_METRICS:
            if "dpt_pseudotime" not in adata_backed.obs:
                raise ValueError(
                    "TRAJ requires adata.obs['dpt_pseudotime']."
                )
            pseudotime = adata_backed.obs["dpt_pseudotime"].copy()

        n_obs = int(adata_backed.n_obs)
        n_vars = int(adata_backed.n_vars)
    finally:
        adata_backed.file.close()

    if obs.isna().any().any():
        raise ValueError(
            "Missing batch/label values: "
            f"{obs.isna().sum().to_dict()}"
        )

    n_batches = int(obs[BATCH_KEY].nunique())
    n_labels = int(obs[LABEL_KEY].nunique())
    if n_batches < 2:
        raise ValueError(
            f"Batch metrics require at least 2 batches; found {n_batches}."
        )

    batch_coverage = obs.groupby(
        LABEL_KEY,
        observed=True,
    )[BATCH_KEY].nunique()
    label_counts = obs.groupby(LABEL_KEY, observed=True).size()

    if "TRAJ" in SELECTED_METRICS:
        finite_pseudotime = pseudotime.dropna()
        if (
            len(finite_pseudotime) < 2
            or finite_pseudotime.nunique() < 2
        ):
            raise ValueError(
                "TRAJ requires at least 2 usable pseudotime values."
            )

    if "ISOLISI" in SELECTED_METRICS:
        min_batch_coverage = int(batch_coverage.min())
        if min_batch_coverage >= n_batches:
            raise ValueError(
                "ISOLISI is enabled, but every label appears in all batches."
            )

    eligible_cpcr_labels = [
        str(label)
        for label in label_counts.index
        if int(label_counts.loc[label]) >= 100
        and int(batch_coverage.loc[label]) >= 2
    ]
    if "CPCR" in SELECTED_METRICS and not eligible_cpcr_labels:
        raise ValueError(
            "CPCR requires a label with at least 100 cells across "
            "at least 2 batches."
        )

    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_batches": n_batches,
        "n_labels": n_labels,
        "cpcr_eligible_labels": len(eligible_cpcr_labels),
    }


validate_dataset_contents()


{'n_obs': 19202,
 'n_vars': 22742,
 'n_batches': 2,
 'n_labels': 15,
 'cpcr_eligible_labels': 12}

In [4]:
OPTUNA_BEST_SOURCE = '/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/results/optuna_multi_metrics/scanorama/cellranger_versions/expanded_metrics_v2/scanorama_cellranger_versions_joint_expanded_metrics_v2_best_params.json'
OPTUNA_STUDY_NAME = 'scanorama_cellranger_versions_joint_expanded_metrics_v2'
OPTUNA_BEST_TRIAL = 131
OPTUNA_BEST_VALUE = 0.7430731944746773
OPTUNA_BEST_PARAMS = {'knn': 50, 'sigma': 59.80765621156959, 'alpha': 0.36097085034101817}

assert tuple(OPTUNA_BEST_PARAMS) == ("knn", "sigma", "alpha")
SCANORAMA_PARAMS = dict(OPTUNA_BEST_PARAMS)

pd.Series(
    {
        "source": OPTUNA_BEST_SOURCE,
        "study": OPTUNA_STUDY_NAME,
        "best_trial": OPTUNA_BEST_TRIAL,
        "best_value": OPTUNA_BEST_VALUE,
        **SCANORAMA_PARAMS,
        "batch_size": BATCH_SIZE,
        "approx": APPROX,
        "union": UNION,
        "internal_dimred": INTERNAL_DIMRED,
        "n_top_hvg": N_TOP_HVG,
    },
    name="best_setting",
)

def check_keys(adata):
    missing_keys = [
        key
        for key in (BATCH_KEY, LABEL_KEY)
        if key not in adata.obs
    ]
    if missing_keys:
        raise ValueError(
            f"Missing keys in adata.obs: {missing_keys}. "
            f"Available keys: {list(adata.obs.columns)}"
        )


def ensure_csr(matrix):
    if sp.issparse(matrix):
        return matrix.tocsr()
    return sp.csr_matrix(matrix)


def split_by_batch(adata):
    batches = adata.obs[BATCH_KEY].astype("category")
    batch_names = list(batches.cat.categories)
    parts = []
    index_parts = []

    for batch_name in batch_names:
        indices = np.where(batches.values == batch_name)[0]
        index_parts.append(indices)
        parts.append(adata[indices, :].copy())

    return parts, batch_names, index_parts


def hvg_subset(adata):
    if "highly_variable" not in adata.var:
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat",
            n_top_genes=N_TOP_HVG,
        )

    hvg_mask = (
        adata.var["highly_variable"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )
    if int(hvg_mask.sum()) == 0:
        raise ValueError("No highly variable genes selected.")
    return adata[:, hvg_mask].copy()


def reassemble_scanorama_embedding(parts, index_parts, n_obs):
    embedding_concat = np.concatenate(
        [part.obsm[EMBED_KEY] for part in parts],
        axis=0,
    )
    concat_indices = np.concatenate(index_parts)
    inverse = np.empty(n_obs, dtype=np.int64)
    inverse[concat_indices] = np.arange(n_obs, dtype=np.int64)
    return embedding_concat[inverse, :]


def run_scanorama_setting(adata, seed):
    np.random.seed(seed)
    check_keys(adata)

    adata_work = hvg_subset(adata.copy())
    adata_work.X = ensure_csr(adata_work.X)
    parts, batch_names, index_parts = split_by_batch(adata_work)

    scanorama.integrate_scanpy(
        parts,
        batch_size=BATCH_SIZE,
        verbose=0,
        ds_names=batch_names,
        dimred=INTERNAL_DIMRED,
        approx=APPROX,
        sigma=float(SCANORAMA_PARAMS["sigma"]),
        alpha=float(SCANORAMA_PARAMS["alpha"]),
        knn=int(SCANORAMA_PARAMS["knn"]),
        union=UNION,
        hvg=None,
        seed=int(seed),
    )

    embedding = reassemble_scanorama_embedding(
        parts,
        index_parts,
        n_obs=adata_work.n_obs,
    )
    if embedding.shape[0] != adata_work.n_obs:
        raise ValueError(
            f"Scanorama output shape {embedding.shape} is incompatible "
            f"with n_obs={adata_work.n_obs}."
        )
    if not np.all(np.isfinite(embedding)):
        raise ValueError(
            "Scanorama produced NaN or Inf in X_scanorama."
        )

    adata_int = adata_work.copy()
    adata_int.obsm[EMBED_KEY] = np.asarray(
        embedding,
        dtype=np.float32,
    )
    return adata_int


In [5]:
def metric_value(metric_series, key):
    if key not in metric_series.index:
        raise KeyError(
            f"Metric {key} not found. "
            f"Available metrics: {list(metric_series.index)}"
        )

    value = metric_series[key]
    if isinstance(value, (list, tuple, np.ndarray)):
        values = np.asarray(value, dtype=float)
        if values.size == 0 or not np.all(np.isfinite(values)):
            raise ValueError(
                f"Metric {key} returned non-finite values: {values}"
            )
        return float(np.mean(values))

    value = float(value)
    if not np.isfinite(value):
        raise ValueError(
            f"Metric {key} returned a non-finite value: {value}"
        )
    return value


def compute_joint_metrics(adata_raw, adata_int):
    sc.pp.neighbors(
        adata_int,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        random_state=0,
    )

    metric_key_map = {
        METRIC_BATCH_KEY: BATCH_KEY,
        METRIC_LABEL_KEY: LABEL_KEY,
    }
    for adata_name, adata_obj in (
        ("adata_raw", adata_raw),
        ("adata_int", adata_int),
    ):
        existing_aliases = [
            alias for alias in metric_key_map if alias in adata_obj.obs
        ]
        if existing_aliases:
            raise ValueError(
                f"Reserved metric obs keys already exist in {adata_name}: "
                f"{existing_aliases}"
            )
        for alias, source_key in metric_key_map.items():
            adata_obj.obs[alias] = adata_obj.obs[source_key].copy()

    metric_df = metrics(
        adata=adata_raw,
        adata_int=adata_int,
        batch_key=METRIC_BATCH_KEY,
        label_key=METRIC_LABEL_KEY,
        embed=EMBED_KEY,
        cluster_key=CLUSTER_KEY,
        verbose=False,
        bami_="BAMI" in SELECTED_METRICS,
        casw_="CASW" in SELECTED_METRICS,
        scgraphm_="SCGRAPHM" in SELECTED_METRICS,
        traj_="TRAJ" in SELECTED_METRICS,
        isolisi_="ISOLISI" in SELECTED_METRICS,
        bras_="BRAS" in SELECTED_METRICS,
        kbet_="KBET" in SELECTED_METRICS,
        cpcr_="CPCR" in SELECTED_METRICS,
    )
    metric_series = metric_df.iloc[:, 0]

    result = {
        key: metric_value(metric_series, key)
        for key in SELECTED_METRICS
    }
    bio_score = float(
        np.mean([result[key] for key in SELECTED_BIO_METRICS])
    )
    batch_score = float(
        np.mean([result[key] for key in SELECTED_BATCH_METRICS])
    )
    joint_score = float(0.6 * bio_score + 0.4 * batch_score)

    if not np.all(np.isfinite([bio_score, batch_score, joint_score])):
        raise ValueError(
            "Aggregation produced a non-finite score: "
            f"bio={bio_score}, batch={batch_score}, "
            f"joint={joint_score}"
        )

    result["bio_score"] = bio_score
    result["batch_score"] = batch_score
    result["joint_score"] = joint_score
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scanorama_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


After v6: 3382 genes
After v8: 3382 genes
Found 3382 genes among all datasets


Cell type silhouette score...


Per cell type PC regression...


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.8207399591135214


optimised clustering against scanorama_metric_label
optimal cluster resolution: 0.8
optimal score: 0.8556727822961531
BRAS...


seed           9.000000
BAMI           0.855673
CASW           0.608765
SCGRAPHM       0.946679
BRAS           0.815146
KBET           0.340077
CPCR           0.775712
bio_score      0.803705
batch_score    0.643645
joint_score    0.739681
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.854531,0.608661,0.946267,0.821459,0.342206,0.775996,0.803153,0.646553,0.740513
1,1,0.857221,0.608894,0.946670,0.809524,0.336216,0.777767,0.804262,0.641169,0.739025
2,2,0.859512,0.608445,0.946500,0.819443,0.332824,0.768908,0.804819,0.640392,0.739048
3,3,0.860357,0.608052,0.946782,0.810808,0.344856,0.770786,0.805064,0.642150,0.739898
4,4,0.843431,0.608829,0.945120,0.815852,0.343722,0.774618,0.799127,0.644731,0.737368
5,5,0.855240,0.608234,0.945445,0.807395,0.341454,0.774603,0.802973,0.641151,0.738244
6,6,0.861673,0.608034,0.945128,0.814163,0.342571,0.782027,0.804945,0.646254,0.741469
7,7,0.850732,0.608084,0.948092,0.814435,0.341849,0.779990,0.802303,0.645425,0.739551
8,8,0.857354,0.608475,0.947352,0.812898,0.336955,0.786970,0.804393,0.645608,0.740879
9,9,0.855673,0.608765,0.946679,0.815146,0.340077,0.775712,0.803705,0.643645,0.739681


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.855572,0.005035,0.843431,0.861673
CASW,0.608447,0.000316,0.608034,0.608894
SCGRAPHM,0.946403,0.000909,0.945120,0.948092
BRAS,0.814112,0.004064,0.807395,0.821459
KBET,0.340273,0.003589,0.332824,0.344856
CPCR,0.776738,0.005017,0.768908,0.786970
bio_score,0.803474,0.001692,0.799127,0.805064
batch_score,0.643708,0.002204,0.640392,0.646553
joint_score,0.739568,0.001162,0.737368,0.741469
